# Week 10 Participation Activity: Improving coding with generative AI

**Qingan Xu**

**Notebook improved:** my Week 5 in-class exercise (`WeeklyModules/Week05/Week_5.ipynb`). It combines daily 2018 stock data for the five FAANG companies (Facebook, Apple, Amazon, Netflix and Google) into one data frame, converts the column types, finds the days with the highest trading volume and reshapes the data into long format.

**Generative AI tool:** Claude (Anthropic). I asked it to recommend improvements to the Python code and to generate an improved notebook to download. Its recommendations come first, then the improved code. The original code is in the appendix at the end, for comparison.

**To run it:** put `aapl.csv`, `amzn.csv`, `fb.csv`, `goog.csv` and `nflx.csv` in the same folder as this notebook, then run all the cells from the top. The notebook creates `faang.csv` in that folder.

## Claude's recommendations

1. **Replace the five copy-pasted read blocks with a loop.** The same two lines were written out once per stock. A loop over the ticker symbols does the work once, so adding a sixth stock means adding one ticker to a list, and a copy-paste slip such as labeling `fb.csv` as `'AMZN'` can't happen.
2. **Set the data folder in one place and check that the files are there.** The original reads `aapl.csv` from whatever folder Jupyter starts in. In my GitHub repository the files are in `data/Week 05 Examples/`, so `Week05/Week_5.ipynb` stops with a `FileNotFoundError` when it's run from its own folder. A `DATA_DIR` setting makes the location easy to change, and a check at the start names every missing file.
3. **Give each step its own cell and heading.** All the steps were in one cell, so re-running the last step meant re-reading all five files, and the results in between couldn't be inspected.
4. **Check the data before relying on it.** `assert` statements confirm that the columns are the expected ones, that no values are missing (converting a column with missing values to integers fails) and that no stock has the same date twice, which the melt in step 2h depends on. If the data ever changes, the notebook stops with a clear message instead of giving a wrong answer.
5. **Make the type conversion explicit and show that it worked.** `format='%Y-%m-%d'` tells `pd.to_datetime` the exact date format, so a date written any other way raises an error instead of being guessed at. `astype('int64')` names the integer size; plain `int` meant 32-bit integers on Windows before NumPy 2.0. A before-and-after table of the column types confirms the change.
6. **Renumber the rows after sorting.** `sort_values(..., ignore_index=True)` numbers the sorted rows 0, 1, 2, … instead of keeping the scrambled labels from before the sort (644, 555, 559, … in the original output).
7. **Let Jupyter display tables instead of printing them.** A data frame on the last line of a cell is shown as a formatted table; `print()` shows plain text.
8. **Name the melted columns.** `var_name='metric'` replaces the default name `variable`, so the long table says what that column holds.
9. **Explain the results.** Short notes after the results say what they show and what to watch out for.

**Kept from the original, because they were already good choices:** `pd.concat(..., ignore_index=True)`, `to_csv(..., index=False)` (no extra column of row numbers in the file), `nlargest()` (simpler and faster than sorting every row) and a comment naming each step.

The improved code gives the same answers as the original: the same seven highest-volume rows and the same 6,275-row long table.

## Setup

In [1]:
from pathlib import Path

import pandas as pd

# The five CSV files need to be in the same folder as this notebook ('.' means this folder).
# If they're somewhere else, point DATA_DIR there, e.g. Path('../../data/Week 05 Examples').
# In Colab, upload them with the Files pane on the left first.
DATA_DIR = Path('.')

# Each stock's file is named after its ticker symbol: AAPL -> aapl.csv, and so on
TICKERS = ['AAPL', 'AMZN', 'FB', 'GOOG', 'NFLX']
FILES = {ticker: DATA_DIR / f'{ticker.lower()}.csv' for ticker in TICKERS}

# Together, date and ticker identify each row: one row per stock per trading day
ID_COLUMNS = ['date', 'ticker']

# Check for every file before reading any, so a missing file gives a clear message
missing = [path.name for path in FILES.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Put these files in {DATA_DIR.resolve()}: {", ".join(missing)}')
print('Found', ', '.join(path.name for path in FILES.values()))

Found aapl.csv, amzn.csv, fb.csv, goog.csv, nflx.csv


## 2b–2c: Read each file and add a ticker column

In [2]:
# One loop replaces the five copy-pasted blocks: read each file and add its ticker column.
# The data frames are kept in a dictionary keyed by ticker instead of five separate variables.
stocks = {ticker: pd.read_csv(path).assign(ticker=ticker) for ticker, path in FILES.items()}

for ticker, stock in stocks.items():
    print(f'{ticker}: {len(stock)} rows, {stock["date"].min()} to {stock["date"].max()}')

AAPL: 251 rows, 2018-01-02 to 2018-12-31
AMZN: 251 rows, 2018-01-02 to 2018-12-31
FB: 251 rows, 2018-01-02 to 2018-12-31
GOOG: 251 rows, 2018-01-02 to 2018-12-31
NFLX: 251 rows, 2018-01-02 to 2018-12-31


## 2d: Combine the five data frames

In [3]:
faang = pd.concat(stocks.values(), ignore_index=True)

# Check the data before the later steps rely on it. If a check fails, the notebook stops
# here and shows the message after the comma, instead of giving a wrong answer later.
expected_columns = ['date', 'open', 'high', 'low', 'close', 'volume', 'ticker']
assert list(faang.columns) == expected_columns, f'Unexpected columns: {list(faang.columns)}'
assert faang.notna().all().all(), 'Some values are missing'
assert not faang.duplicated(ID_COLUMNS).any(), 'A stock has the same date more than once'

print(faang.shape)
faang.head()

(1255, 7)


,date,open,high,low,close,volume,ticker
0,2018-01-02,166.9271,169.0264,166.0442,168.9872,25555934,AAPL
1,2018-01-03,169.2521,171.2337,168.6929,168.9578,29517899,AAPL
2,2018-01-04,169.2619,170.1742,168.8106,169.7426,22434597,AAPL
3,2018-01-05,170.1448,172.0381,169.7622,171.6751,23660018,AAPL
4,2018-01-08,171.0375,172.2736,170.6255,171.0375,20567766,AAPL


## 2e: Save the combined data to a CSV file

The file is saved before the type conversion, following the order of the exercise. A CSV file stores everything as text, so when reading it back, use `pd.read_csv('faang.csv', parse_dates=['date'])` to get the dates as datetimes again.

In [4]:
faang.to_csv('faang.csv', index=False)   # index=False leaves the row numbers out of the file
print(f'Saved {len(faang):,} rows to faang.csv')

Saved 1,255 rows to faang.csv


## 2f: Convert the data types and sort by date and ticker

In [5]:
types_before = faang.dtypes

faang = faang.assign(
    # format= gives the exact date format, so pandas doesn't guess and
    # any date written differently raises an error
    date=pd.to_datetime(faang['date'], format='%Y-%m-%d'),
    # 'int64' names the integer size; plain int meant 32-bit on Windows before NumPy 2.0
    volume=faang['volume'].astype('int64'),
).sort_values(ID_COLUMNS, ignore_index=True)   # ignore_index renumbers the sorted rows 0, 1, 2, ...

# Column types before and after the conversion
pd.DataFrame({'before': types_before, 'after': faang.dtypes})

,before,after
date,str,datetime64[us]
open,float64,float64
high,float64,float64
low,float64,float64
close,float64,float64
volume,int64,int64
ticker,str,str


`date` is now a datetime instead of text. `read_csv` had already read `volume` as `int64` because every volume in the files is a whole number, so that conversion changes nothing here. It's kept because the exercise asks for it, and it makes sure `volume` is an integer column even if a file stores the volumes as, say, `25555934.0`.

## 2g: The seven rows with the highest volume

In [6]:
# nlargest() is simpler and faster than sorting all 1,255 rows and taking the first seven
top_7_volume = faang.nlargest(7, 'volume')[['date', 'ticker', 'volume']]
top_7_volume

,date,ticker,volume
712,2018-07-26,FB,169803668
267,2018-03-20,FB,129851768
287,2018-03-26,FB,126116634
272,2018-03-21,FB,106598834
910,2018-09-21,AAPL,96246748
1225,2018-12-21,AAPL,95744384
1060,2018-11-02,AAPL,91328654


Facebook had the four busiest days. July 26 was the day after Facebook's second-quarter earnings report, when its stock fell about 19%, and the three March days came just after the Cambridge Analytica scandal broke on March 17.

Volume counts shares, not dollars, so this ranking favors lower-priced stocks. Each stock's median closing price and daily volume show why Amazon and Google don't appear: their shares cost over $1,000, so far fewer of them changed hands each day.

In [7]:
# Typical (median) closing price and number of shares traded a day for each stock
faang.groupby('ticker')[['close', 'volume']].median().astype({'volume': 'int64'})

,close,volume
ticker,,
AAPL,184.2231,31663096
AMZN,1624.8900,4954988
FB,174.7000,21860931
GOOG,1103.9800,1531299
NFLX,321.3000,10428621


## 2h: Melt the data into long format

In [8]:
# date and ticker stay as columns; the other five columns are stacked into two new ones:
# metric (open, high, low, close or volume) and value
faang_melted = faang.melt(id_vars=ID_COLUMNS, var_name='metric', value_name='value')

values_per_row = faang.shape[1] - len(ID_COLUMNS)   # the five metrics
assert len(faang_melted) == len(faang) * values_per_row
print(faang_melted.shape)
faang_melted.head()

(6275, 4)


,date,ticker,metric,value
0,2018-01-02,AAPL,open,166.9271
1,2018-01-02,AMZN,open,1172.0000
2,2018-01-02,FB,open,177.6800
3,2018-01-02,GOOG,open,1048.3400
4,2018-01-02,NFLX,open,196.1000


Each row of `faang` became five rows, one for each metric: 1,255 × 5 = 6,275 rows. Because the `value` column now holds both prices and share counts, pandas stores the volumes as decimal numbers (`float64`) too.

## Appendix: the original code

This was the only code cell in `WeeklyModules/Week05/Week_5.ipynb`. It's shown as text here, so it doesn't run.

```python
import pandas as pd

# 2b & 2c: Read each file into a dataframe and add a 'ticker' column
aapl = pd.read_csv('aapl.csv')
aapl['ticker'] = 'AAPL'

amzn = pd.read_csv('amzn.csv')
amzn['ticker'] = 'AMZN'

fb = pd.read_csv('fb.csv')
fb['ticker'] = 'FB'

goog = pd.read_csv('goog.csv')
goog['ticker'] = 'GOOG'

nflx = pd.read_csv('nflx.csv')
nflx['ticker'] = 'NFLX'

# 2d: Append each file together into a single dataframe
faang = pd.concat([aapl, amzn, fb, goog, nflx], ignore_index=True)

# 2e: Save the results in a CSV file
faang.to_csv('faang.csv', index=False)

# 2f: Convert 'date' to datetime, 'volume' to integer, and sort
faang['date'] = pd.to_datetime(faang['date'])
faang['volume'] = faang['volume'].astype(int)
faang = faang.sort_values(by=['date', 'ticker'])

# 2g: Find the seven rows with the highest value for volume
top_7_volume = faang.nlargest(7, 'volume')
print(top_7_volume[['date', 'ticker', 'volume']])

# 2h: Melt the dataframe using 'date' and 'ticker' as key identifiers
faang_melted = faang.melt(id_vars=['date', 'ticker'])
print(faang_melted.head())
```